# Notebook 2: Efficient Representation And Circuit Validation

This notebook compares `phase_b.pt` and `phase_c.pt` side by side with four faster validation experiments:

1. `neighbor_agreement`: does local neighbor structure in `z` match `q`?
2. `activation_probe`: can `z` linearly decode same-layer pooled backbone state?
3. `discovery_pilot`: can `z` support circuit-like grouping on a small deterministic node panel?
4. `topk_interventions`: do the strongest pilot circuits cause larger member-specific effects than controls?

Phase C is better when it preserves useful predictive structure while improving one or more of these downstream checks.


In [ ]:
# Colab-first setup: clone/update the repo, install it, and mount Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Running outside Colab; using the current working tree.")


In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.evaluation.efficient_validation import (
    EFFICIENT_EXPERIMENT_IDS,
    run_activation_probe_experiment,
    run_discovery_pilot_experiment,
    run_neighbor_agreement_experiment,
    run_topk_intervention_experiment,
)
from flow_circuits.training import load_components_from_checkpoint


## Config

Change the single `EXPERIMENTS` line below to run all four experiments or any subset.
The notebook always compares `phase_b.pt` and `phase_c.pt` and never retrains the flow model.


In [ ]:
from pathlib import Path

RUN_MODE = "debug"  # "debug" or "full"
CONFIG_NAME = "resnet18_aligned"
EXPERIMENTS = "all"
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits")
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb02_efficient_validation" / CONFIG_NAME

PHASE_B_CHECKPOINT = EXPERIMENT_ROOT / "phase_b.pt"
PHASE_C_CHECKPOINT = EXPERIMENT_ROOT / "phase_c.pt"


In [ ]:
import json
import time
from pathlib import Path

import torch

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    import pandas as pd
except ImportError:
    pd = None

MODEL_ORDER = [("phase_b", "Phase B"), ("phase_c", "Phase C")]
EXPERIMENT_LABELS = {
    "neighbor_agreement": "Neighbor Agreement",
    "activation_probe": "Activation Probe",
    "discovery_pilot": "Discovery Pilot",
    "topk_interventions": "Top-k Interventions",
}

if EXPERIMENTS == "all":
    SELECTED_EXPERIMENTS = list(EFFICIENT_EXPERIMENT_IDS)
else:
    SELECTED_EXPERIMENTS = [str(name) for name in EXPERIMENTS]
invalid = sorted(set(SELECTED_EXPERIMENTS) - set(EFFICIENT_EXPERIMENT_IDS))
if invalid:
    raise ValueError(f"Unknown experiments: {invalid}. Valid ids: {EFFICIENT_EXPERIMENT_IDS}")

if not PHASE_B_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_B_CHECKPOINT}")
if not PHASE_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_C_CHECKPOINT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR = OUTPUT_DIR / "comparisons"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_BY_MODE = {
    "debug": {
        "neighbor_agreement": {"max_images": 1000, "anchor_images": 128, "topk": 20},
        "activation_probe": {"fit_max_images": 1000, "eval_max_images": 500, "ridge_alpha": 1.0},
        "discovery_pilot": {"max_images": 1000, "nodes_per_layer": 1, "bootstrap_iterations": 2},
        "topk_interventions": {"max_images": 1000, "topk": 2, "min_image_set_size": 25},
    },
    "full": {
        "neighbor_agreement": {"max_images": 5000, "anchor_images": 256, "topk": 20},
        "activation_probe": {"fit_max_images": 4000, "eval_max_images": 1000, "ridge_alpha": 1.0},
        "discovery_pilot": {"max_images": 5000, "nodes_per_layer": 2, "bootstrap_iterations": 5},
        "topk_interventions": {"max_images": 5000, "topk": 5, "min_image_set_size": 25},
    },
}
if RUN_MODE not in SETTINGS_BY_MODE:
    raise ValueError(f"RUN_MODE must be one of {sorted(SETTINGS_BY_MODE)}")

def _format_seconds(seconds):
    if seconds is None:
        return "?"
    seconds = int(max(0, round(seconds)))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"

def _progress_logger(**event):
    stamp = time.strftime("%H:%M:%S")
    label = "Phase C" if event["checkpoint_tag"] == "phase_c" else "Phase B"
    stage = event["stage"].replace("_", " ")
    total = event.get("total")
    if total is None:
        counter = stage
    else:
        counter = f"{stage} {event['completed']}/{total}"
    elapsed = _format_seconds(event.get("elapsed_seconds"))
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    if eta is None:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | {message}", flush=True)
    else:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | ETA {_format_seconds(eta)} | {message}", flush=True)

def _cache_path(tag, experiment_id):
    return OUTPUT_DIR / tag / f"{experiment_id}.json"

def _load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def _should_run(experiment_id):
    return experiment_id in SELECTED_EXPERIMENTS

def _write_summary_table(rows, *, title):
    print(title)
    if not rows:
        print("  No rows to display.")
        return
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)

def _bar_compare(title, metric_name, phase_b_value, phase_c_value):
    if plt is None:
        return
    fig, ax = plt.subplots(figsize=(4.5, 3.0))
    ax.bar(["Phase B", "Phase C"], [phase_b_value, phase_c_value], color=["#5975A4", "#CC8963"])
    ax.set_title(title)
    ax.set_ylabel(metric_name)
    plt.show()

RESULTS = {}
PILOT_RESULTS = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"Run mode      : {RUN_MODE}")
print(f"Config        : {CONFIG_NAME}")
print(f"Experiments   : {SELECTED_EXPERIMENTS}")
print(f"Phase B ckpt  : {PHASE_B_CHECKPOINT}")
print(f"Phase C ckpt  : {PHASE_C_CHECKPOINT}")
print(f"Output dir    : {OUTPUT_DIR}")

components_by_tag = {
    "phase_b": load_components_from_checkpoint(PHASE_B_CHECKPOINT, device=device),
    "phase_c": load_components_from_checkpoint(PHASE_C_CHECKPOINT, device=device),
}
base_config = components_by_tag["phase_b"].config
loader_batch_size = max(
    int(base_config["data"]["batch_size"]),
    int(base_config.get("discovery", {}).get("batch_size", 0) or 0),
    int(base_config.get("interventions", {}).get("batch_size", 0) or 0),
)
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=loader_batch_size,
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)

def _run_or_load(tag, experiment_id, run_fn):
    cache_path = _cache_path(tag, experiment_id)
    if cache_path.exists() and not FORCE_RERUN:
        print(f"Loading cached {experiment_id} for {tag}: {cache_path}")
        return _load_json(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Running {experiment_id} for {tag}; cache -> {cache_path}")
    return run_fn(cache_path)

def _ensure_discovery_pilot(tag):
    if tag in PILOT_RESULTS and not FORCE_RERUN:
        return PILOT_RESULTS[tag]
    settings = SETTINGS_BY_MODE[RUN_MODE]["discovery_pilot"]
    shared_panel = None
    if tag == "phase_c":
        shared_panel = _ensure_discovery_pilot("phase_b")["selected_node_panel"]
    def _runner(cache_path):
        return run_discovery_pilot_experiment(
            components_by_tag[tag],
            loaders["discovery"],
            device=device,
            checkpoint_tag=tag,
            max_images=settings["max_images"],
            nodes_per_layer=settings["nodes_per_layer"],
            bootstrap_iterations=settings["bootstrap_iterations"],
            node_panel=shared_panel,
            output_path=cache_path,
            progress_callback=_progress_logger,
        )
    result = _run_or_load(tag, "discovery_pilot", _runner)
    PILOT_RESULTS[tag] = result
    return result


## Experiment 1: Neighbor Agreement

This asks whether each `(layer, cell)` token preserves the same nearest-neighbor image structure as the frozen future descriptor `q`.
Higher `recall@20` and `Jaccard@20` mean `z` is preserving more of the same local geometry.


In [ ]:
if _should_run("neighbor_agreement"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["neighbor_agreement"]
    neighbor_results = {}
    for tag, _label in MODEL_ORDER:
        neighbor_results[tag] = _run_or_load(
            tag,
            "neighbor_agreement",
            lambda cache_path, tag=tag: run_neighbor_agreement_experiment(
                components_by_tag[tag],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                max_images=settings["max_images"],
                anchor_images=settings["anchor_images"],
                topk=settings["topk"],
                seed=base_config["data"].get("seed", 0),
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["neighbor_agreement"] = neighbor_results
else:
    print("Skipping neighbor_agreement.")


In [ ]:
if "neighbor_agreement" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["neighbor_agreement"][tag]["summary"]
        rows.append(
            {
                "checkpoint": label,
                "mean_recall_at_k": summary["mean_recall_at_k"],
                "mean_jaccard_at_k": summary["mean_jaccard_at_k"],
            }
        )
    _write_summary_table(rows, title="Neighbor agreement summary")
    _bar_compare(
        title="Neighbor Agreement",
        metric_name="mean recall@k",
        phase_b_value=RESULTS["neighbor_agreement"]["phase_b"]["summary"]["mean_recall_at_k"],
        phase_c_value=RESULTS["neighbor_agreement"]["phase_c"]["summary"]["mean_recall_at_k"],
    )


## Experiment 2: Activation Probe

This fits a per-layer ridge probe from `z[l, i]` back to the pooled same-layer backbone state.
Higher cosine and `R^2` mean `z` is retaining more local state information.


In [ ]:
if _should_run("activation_probe"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["activation_probe"]
    activation_results = {}
    for tag, _label in MODEL_ORDER:
        activation_results[tag] = _run_or_load(
            tag,
            "activation_probe",
            lambda cache_path, tag=tag: run_activation_probe_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                eval_max_images=settings["eval_max_images"],
                ridge_alpha=settings["ridge_alpha"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["activation_probe"] = activation_results
else:
    print("Skipping activation_probe.")


In [ ]:
if "activation_probe" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["activation_probe"][tag]["summary"]
        rows.append(
            {
                "checkpoint": label,
                "mean_cosine": summary["mean_cosine"],
                "mean_r2": summary["mean_r2"],
            }
        )
    _write_summary_table(rows, title="Activation probe summary")
    _bar_compare(
        title="Activation Probe",
        metric_name="mean cosine",
        phase_b_value=RESULTS["activation_probe"]["phase_b"]["summary"]["mean_cosine"],
        phase_c_value=RESULTS["activation_probe"]["phase_c"]["summary"]["mean_cosine"],
    )


## Experiment 3: Discovery Pilot

This is a lighter circuit-discovery probe: it scores `q`-dispersion, picks a small deterministic node panel, and runs discovery only there.
Higher circuit count, stability, and depth span suggest more reusable circuit structure.


In [ ]:
if _should_run("discovery_pilot"):
    discovery_results = {tag: _ensure_discovery_pilot(tag) for tag, _label in MODEL_ORDER}
    RESULTS["discovery_pilot"] = discovery_results
else:
    print("Skipping discovery_pilot.")


In [ ]:
if "discovery_pilot" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["discovery_pilot"][tag]["summary"]
        rows.append(
            {
                "checkpoint": label,
                "n_circuits": summary["n_circuits"],
                "mean_cluster_stability": summary["mean_cluster_stability"],
                "mean_active_node_depth_span": summary["mean_active_node_depth_span"],
            }
        )
    _write_summary_table(rows, title="Discovery pilot summary")
    _bar_compare(
        title="Discovery Pilot",
        metric_name="pilot circuits",
        phase_b_value=RESULTS["discovery_pilot"]["phase_b"]["summary"]["n_circuits"],
        phase_c_value=RESULTS["discovery_pilot"]["phase_c"]["summary"]["n_circuits"],
    )


## Experiment 4: Top-k Interventions

This loads or generates the pilot circuits, keeps only the strongest few, and runs held-out member-vs-control interventions.
Stronger member-specific effects and more validated circuits support a more causally specific representation.


In [ ]:
if _should_run("topk_interventions"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["topk_interventions"]
    topk_results = {}
    for tag, _label in MODEL_ORDER:
        pilot_artifact = _ensure_discovery_pilot(tag)
        topk_results[tag] = _run_or_load(
            tag,
            "topk_interventions",
            lambda cache_path, tag=tag, pilot_artifact=pilot_artifact: run_topk_intervention_experiment(
                components_by_tag[tag],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                alpha=base_config.get("interventions", {}).get("alpha", 0.05),
                topk=settings["topk"],
                min_image_set_size=settings["min_image_set_size"],
                max_images=settings["max_images"],
                circuits_artifact=pilot_artifact,
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["topk_interventions"] = topk_results
else:
    print("Skipping topk_interventions.")


In [ ]:
if "topk_interventions" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["topk_interventions"][tag]["summary"]
        selection = RESULTS["topk_interventions"][tag]["selection"]
        rows.append(
            {
                "checkpoint": label,
                "selected_circuits": selection["n_selected_circuits"],
                "member_specific_count": summary["member_specific_count"],
                "validated_count": summary["validated_count"],
                "mean_member_delta_margin": summary["mean_member_delta_margin"],
            }
        )
    _write_summary_table(rows, title="Top-k intervention summary")
    _bar_compare(
        title="Top-k Interventions",
        metric_name="member-specific circuits",
        phase_b_value=RESULTS["topk_interventions"]["phase_b"]["summary"]["member_specific_count"],
        phase_c_value=RESULTS["topk_interventions"]["phase_c"]["summary"]["member_specific_count"],
    )


## Final Comparison

This writes one compact comparison summary across all experiments selected in this run.


In [ ]:
summary_rows = []
for experiment_id in SELECTED_EXPERIMENTS:
    if experiment_id not in RESULTS:
        continue
    phase_b_result = RESULTS[experiment_id]["phase_b"]
    phase_c_result = RESULTS[experiment_id]["phase_c"]
    if experiment_id == "neighbor_agreement":
        summary_rows.append(
            {
                "experiment": "Neighbor Agreement",
                "phase_b": phase_b_result["summary"]["mean_recall_at_k"],
                "phase_c": phase_c_result["summary"]["mean_recall_at_k"],
                "metric": "mean_recall_at_k",
            }
        )
    elif experiment_id == "activation_probe":
        summary_rows.append(
            {
                "experiment": "Activation Probe",
                "phase_b": phase_b_result["summary"]["mean_cosine"],
                "phase_c": phase_c_result["summary"]["mean_cosine"],
                "metric": "mean_cosine",
            }
        )
    elif experiment_id == "discovery_pilot":
        summary_rows.append(
            {
                "experiment": "Discovery Pilot",
                "phase_b": phase_b_result["summary"]["n_circuits"],
                "phase_c": phase_c_result["summary"]["n_circuits"],
                "metric": "n_circuits",
            }
        )
    elif experiment_id == "topk_interventions":
        summary_rows.append(
            {
                "experiment": "Top-k Interventions",
                "phase_b": phase_b_result["summary"]["member_specific_count"],
                "phase_c": phase_c_result["summary"]["member_specific_count"],
                "metric": "member_specific_count",
            }
        )

summary_payload = {
    "run_mode": RUN_MODE,
    "config_name": CONFIG_NAME,
    "selected_experiments": SELECTED_EXPERIMENTS,
    "summary_rows": summary_rows,
}
(COMPARISON_DIR / "summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
if pd is not None:
    pd.DataFrame(summary_rows).to_csv(COMPARISON_DIR / "summary.csv", index=False)
_write_summary_table(summary_rows, title="Efficient validation comparison summary")
print(f"Summary JSON : {COMPARISON_DIR / 'summary.json'}")
if pd is not None:
    print(f"Summary CSV  : {COMPARISON_DIR / 'summary.csv'}")
